# Day 41 — Practice Questions

## Support Vector Machines

**Boston University · Data Science Program**

---

**12 questions in four parts.**

| Part | | Questions |
|---|---|---|
| **A** | The street and the points that hold it open | 1 – 3 |
| **B** | The `C` knob | 4 – 6 |
| **C** | Kernels and `gamma` | 7 – 9 |
| **D** | Judgement | 10 – 12 |

Run the setup cell first. Every question is self-contained after that.

> Replace every `...` with your own code. Where a question asks you to *explain*,
> write two or three sentences in the markdown cell — the explanation is the point of
> the question, not the number.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.simplefilter("ignore")

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV

CV = StratifiedKFold(5, shuffle=True, random_state=0)
TEAL, CORAL, INK = "#1F9E86", "#E05C45", "#131F33"

def show(X, y, model=None, title="", ax=None, support=True):
    """Points, and -- if a model is given -- its street."""
    ax = ax or plt.gca()
    if model is not None:
        pad = 0.6
        gx, gy = np.meshgrid(
            np.linspace(X[:, 0].min() - pad, X[:, 0].max() + pad, 320),
            np.linspace(X[:, 1].min() - pad, X[:, 1].max() + pad, 320))
        Z = model.decision_function(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)
        ax.contourf(gx, gy, Z, levels=[-1, 1], colors=["#FFF1D6"])
        ax.contour(gx, gy, Z, levels=[-1, 0, 1], colors=["#7B8AA1", INK, "#7B8AA1"],
                   linestyles=["--", "-", "--"], linewidths=[1.3, 2.4, 1.3])
        if support and hasattr(model, "support_vectors_"):
            ax.scatter(*model.support_vectors_.T, s=230, facecolors="none",
                       edgecolors="#A86F00", linewidths=2.4, zorder=6)
    for cls, col, mk in [(0, TEAL, "o"), (1, CORAL, "s")]:
        m = y == cls
        ax.scatter(X[m, 0], X[m, 1], c=col, marker=mk, s=70, zorder=5,
                   edgecolors="white", linewidths=1.4)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_title(title, fontsize=12)

# ---- the dataset for Part A ------------------------------------------------
# 40 students. Two columns, both out of 100, so distances mean something
# without scaling: past-paper marks, and attendance percentage. Did they pass?
r = np.random.default_rng(7)
fail = np.c_[r.normal(40, 6.5, 20).clip(0, 100), r.normal(54, 6.5, 20).clip(0, 100)]
pas  = np.c_[r.normal(74, 6.5, 20).clip(0, 100), r.normal(84, 6.5, 20).clip(0, 100)]
XP = np.vstack([fail, pas])
yP = np.r_[np.zeros(20), np.ones(20)]

print("Part A data:", XP.shape, " passed:", int(yP.sum()))
print("marks     ", XP[:, 0].min().round(1), "to", XP[:, 0].max().round(1))
print("attendance", XP[:, 1].min().round(1), "to", XP[:, 1].max().round(1))

---
# Part A · The street and the points that hold it open

## Question 1

Fit a **linear** SVM to `XP, yP` with `C=1e6` (a huge `C` means "do not allow any point
inside the street"), draw it with `show`, and print:

- how many training points there are
- how many of them are support vectors
- what percentage that is

In [ ]:
svm = SVC(kernel=..., C=...).fit(..., ...)

show(XP, yP, svm, "linear SVM on the exam data")
plt.show()

print("training points :", ...)
print("support vectors :", ...)
print("percentage      :", ...)

## Question 2

Test the claim from the slides. Keep **only** the support vectors, refit the same model
on just those rows, and check whether the two models predict the same thing on all 40
original rows.

Then explain in your own words why this works — and when it would *not*.

In [ ]:
keep_X = XP[...]
keep_y = yP[...]
small  = SVC(kernel="linear", C=1e6).fit(..., ...)

print("rows used by the small model:", len(keep_X))
print("same predictions on all 40 rows:", ...)

## Question 3

Here are two candidate straight lines. A line predicts "pass" when
`w1 * marks + w2 * attendance + b > 0`.

| line | what it does | `w1` | `w2` | `b` |
|---|---|---|---|---|
| `L1` | pass if marks are above 56.3 | 1.0 | 0.0 | −56.327 |
| `L2` | pass if attendance is above 66.0 | 0.0 | 1.0 | −65.973 |

**Both of them classify all 40 students correctly.** Accuracy cannot tell them apart.

Write a function that measures the **width of the street** a line gets — the distance
from the line to the nearest point, doubled — then report the width for `L1`, for `L2`,
and for the SVM's own line from Question 1. Which would an SVM choose, and why does the
width matter when the accuracy is identical?

In [ ]:
def street_width(w1, w2, b, X, y):
    w = np.array([w1, w2])
    # signed distance from the line to every point
    d = (X @ w + b) / np.linalg.norm(w)
    # flip the failing group so that "on the correct side" is always positive
    d = np.where(y == 1, d, -d)
    return ...        # twice the distance to the nearest point

for name, (w1, w2, b) in [("L1", (1.0, 0.0, -56.327)),
                          ("L2", (0.0, 1.0, -65.973))]:
    print(name, " street width", round(street_width(w1, w2, b, XP, yP), 3))

---
# Part B · The `C` knob

Now make the problem realistic. The cell below pushes the two groups together so that
**no straight line can separate them**.

In [ ]:
r2 = np.random.default_rng(99)
XB = np.vstack([np.c_[r2.normal(4.6, 1.9, 30), r2.normal(47, 9.5, 30)],
                np.c_[r2.normal(7.0, 1.9, 30), r2.normal(57, 9.5, 30)]])
yB = np.r_[np.zeros(30), np.ones(30)]

show(XB, yB, title="overlapping: no line can separate these")
plt.show()

## Question 4

Fit a linear SVM at `C = 0.001`, `C = 0.1` and `C = 100`. For each one print

- the **width** of the street, which is `2 / norm(model.coef_[0])`
- how many points end up **inside** it (a point is inside when `abs(model.decision_function(X)) < 1`)

and draw all three side by side. Which direction does the street move as `C` grows?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
for ax, c in zip(axes, [..., ..., ...]):
    m = SVC(kernel="linear", C=c).fit(XB, yB)
    width  = ...
    inside = ...
    show(XB, yB, m, f"C = {c}  (width {width:.2f})", ax=ax)
    print(f"C = {c:>7}   width {width:>7.3f}   inside {inside:>2}")
plt.tight_layout(); plt.show()

## Question 5

Accuracy on the training data is not how you choose `C`. For each value in
`[0.001, 0.01, 0.1, 1, 10, 100, 1000]` print the **training** accuracy and the
**cross-validated** accuracy side by side.

Which value would you ship, and what do the two columns together tell you?

In [ ]:
print(f"{'C':>8}  {'train':>7}  {'CV':>7}")
for c in [0.001, 0.01, 0.1, 1, 10, 100, 1000]:
    tr = SVC(kernel="linear", C=c).fit(XB, yB).score(..., ...)
    cv = cross_val_score(SVC(kernel="linear", C=c), ..., ..., cv=CV).mean()
    print(f"{c:>8}  {tr:>7.4f}  {cv:>7.4f}")

## Question 6

Two students describe `C` to each other:

> **Aarav:** "Big `C` means a big street. The letter C is for *capacity*."
>
> **Divya:** "Big `C` means a strict model. It is the *cost* of a mistake."

One of them is right. Say which, and support your answer with a number from your own
output rather than from memory.

In [ ]:
# produce the evidence, then write your answer in the markdown cell below
...

*Your answer:*

---
# Part C · Kernels and `gamma`

Two interleaving crescents. No straight line is going to help here.

In [ ]:
XM, yM = make_moons(n_samples=200, noise=0.22, random_state=41)
show(XM, yM, title="make_moons: two crescents")
plt.show()

## Question 7

Compare a **linear** SVM and an **rbf** SVM on the moons, both inside a
`make_pipeline(StandardScaler(), ...)`, using `cross_val_score`. Print both scores and
the baseline, then draw both boundaries.

In [ ]:
lin = make_pipeline(StandardScaler(), SVC(kernel=...))
rbf = make_pipeline(StandardScaler(), SVC(kernel=...))

print("baseline :", ...)
print("linear   :", round(cross_val_score(lin, XM, yM, cv=CV).mean(), 4))
print("rbf      :", ...)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
show(XM, yM, SVC(kernel="linear").fit(XM, yM), "linear", ax=axes[0], support=False)
show(XM, yM, ..., "rbf", ax=axes[1], support=False)
plt.tight_layout(); plt.show()

## Question 8

Sweep `gamma` over `[0.001, 0.01, 0.1, 1, 10, 100]` with `C=1`, and for each value print
the **training** accuracy and the **cross-validated** accuracy.

Then name the value where the model starts memorising, and say which number in your
output gave it away.

In [ ]:
print(f"{'gamma':>8}  {'train':>7}  {'CV':>7}")
for g in [0.001, 0.01, 0.1, 1, 10, 100]:
    pipe = make_pipeline(StandardScaler(), SVC(C=1, gamma=g))
    tr = pipe.fit(XM, yM).score(XM, yM)
    cv = ...
    print(f"{g:>8}  {tr:>7.4f}  {cv:>7.4f}")

## Question 9

`C` and `gamma` interact, so they have to be searched **together**. Run a
`GridSearchCV` over

- `C` in `[0.1, 1, 10, 100]`
- `gamma` in `[0.001, 0.01, 0.1, 1]`

inside a scaled pipeline. Print the best pair, its score, and how many model fits the
search performed.

In [ ]:
pipe = make_pipeline(StandardScaler(), SVC())
grid = GridSearchCV(pipe, {"svc__C": ..., "svc__gamma": ...}, cv=CV, n_jobs=-1)
grid.fit(XM, yM)

print("best pair :", grid.best_params_)
print("best score:", round(grid.best_score_, 4))
print("fits performed:", ...)

---
# Part D · Judgement

## Question 10

Back to the exam data from Part A, but with a third column bolted on: the student's
**family income in rupees**, which is pure noise and has nothing to do with whether they
pass.

Fit four models with `cross_val_score` and print all four scores:

1. the baseline (always guess the bigger group)
2. `SVC()` on the raw columns
3. `SVC()` inside a `StandardScaler` pipeline
4. `RandomForestClassifier(n_estimators=200, random_state=0)` on the raw columns

Then explain the pattern you see — in particular, why model 4 behaves differently from
model 2.

In [ ]:
r3 = np.random.default_rng(777)
income = r3.normal(480000, 160000, len(XP)).clip(60000, 1200000)   # pure noise
XD = np.c_[XP, income]

print("columns: marks, attendance, income")
print("column ranges:", XD.min(0).round(0), "to", XD.max(0).round(0))

In [ ]:
print("baseline          :", ...)
print("SVC, raw          :", ...)
print("SVC, scaled       :", ...)
print("forest, raw       :", ...)

## Question 11

Time the two model families as the number of rows grows. Fit a scaled `SVC` and a
200-tree `RandomForestClassifier` on 500, 2000, 8000 and 24000 rows of the moons data,
and print how long each `fit` takes.

Careful here: do **not** just read off which model is faster. Compare how each one's time
*grows* as the rows multiply, and say what that means for a million rows.

In [ ]:
import time

for n in [500, 2000, 8000, 24000]:
    Xn, yn = make_moons(n_samples=n, noise=0.25, random_state=0)
    for name, model in [("svm   ", make_pipeline(StandardScaler(), SVC())),
                        ("forest", RandomForestClassifier(n_estimators=200, random_state=0))]:
        t = time.perf_counter()
        model.fit(..., ...)
        print(f"n = {n:>6}  {name}  {time.perf_counter() - t:.3f}s")
    print()

## Question 12

For each situation, say whether you would reach for an **SVM** or a **random forest**,
and give one sentence of justification.

1. 180 tumour samples, 20,000 gene expression columns, and you want the best accuracy you can get.
2. 4 million rows of clickstream data, retrained nightly.
3. A loan committee needs to be told, for each rejected application, which factors drove the decision.
4. 600 rows, 8 numeric columns, two groups that look well separated on a scatter plot.
5. A table of 50,000 rows mixing numbers with text categories such as city and product type, with missing values scattered through it.

*Your answers:*

1.
2.
3.
4.
5.

---

## Submission

Run **Cell → Run All** on a fresh kernel and confirm there are no errors, then upload the
notebook. Questions 6, 8, 10, 11 and 12 are marked mostly on the explanation — a correct
number with no reasoning gets little credit.

Solutions will be released with Day 42.